# Exercice 17 — Recherche d'hyperparamètres

In [14]:
from sklearn.datasets import make_classification

from sklearn.model_selection import train_test_split

from sklearn.tree import DecisionTreeClassifier 

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score ,f1_score

from sklearn.tree import plot_tree

from sklearn.model_selection import  GridSearchCV , RandomizedSearchCV

import matplotlib.pyplot as plt

import pandas as pd

import time


### Réutilisation du dataset et le RandomForestClassifier de l'exercice 8

Generation du dataset

In [15]:
x , y = make_classification(n_samples=800, n_features=15, n_informative=8, n_redundant=3, n_classes=4, random_state=1)

Ajoute de nom de feature

In [16]:
feature_name = [f"feature_{i}" for i in range(1,16)]

x = pd.DataFrame(x,columns=feature_name)


Separation du train et du test

In [17]:
x_train , x_test , y_train , y_test =  train_test_split(x, y, test_size=0.25)

Entrainement du RFC

In [18]:
model = RandomForestClassifier(n_estimators=200, random_state=1)

model.fit(x_train,y_train)

model.score(x_train,y_train)

model.score(x_test,y_test)

pred =  model.predict(x_test)

### Definition des grille pour GridSearch

In [19]:
n_estimators=[50, 100, 200, 300]

max_depth=[3, 5, 7, 10, None]

min_samples_leaf=[1, 2, 4, 8]

### GridSearchCV sur cette grille

In [26]:
debut = time.perf_counter()

chercher =  GridSearchCV(

    estimator=model,

    param_grid={

        "n_estimators" : n_estimators ,

        "max_depth" : max_depth ,

        "min_samples_leaf" : min_samples_leaf

    },

    cv= 5 ,

    scoring="f1_macro"


)

chercher.fit(x_train,y_train)

fin = time.perf_counter()

t = fin - debut

print(f"Temps : {t} s ")


Temps : 308.4696078880006 s 


### RandomizedSearchCV sur cette grille

In [34]:
debut = time.perf_counter()

chercher_r =  RandomizedSearchCV(

    estimator=model,

    param_distributions={

        "n_estimators" : n_estimators ,

        "max_depth" : max_depth ,

        "min_samples_leaf" : min_samples_leaf

    },

    cv= 5 ,

    scoring="f1_macro",

    random_state=42

)

chercher_r.fit(x_train,y_train)

fin = time.perf_counter()

t_r = fin - debut

print(f"Temps : {t_r} s ")


Temps : 38.67595620400243 s 


### Comparaison dans un tableau Pandas : meilleur score obtenu, meilleurs hyperparamètres trouvés, et temps de calcul pour les deux approches

In [40]:
df =  pd.DataFrame({

    "Methode" : ["GridSearchCV","RandomizedSearchCV"],

    "best_score" :  [chercher.best_score_ ,  chercher_r.best_score_],
    
    "best_params" : [chercher.best_params_ , chercher_r.best_params_],

    "temp_exec" :  [t , t_r]

})

df

,Methode,best_score,best_params,temp_exec
0,GridSearchCV,0.66012,"{'n_estimators': 200, 'min_samples_leaf': 2, '...",36.694966
1,RandomizedSearchCV,0.66012,"{'n_estimators': 200, 'min_samples_leaf': 2, '...",38.675956


Conclusion :

GridSearchCV permet d'explorer exhaustivement toutes les combinaisons d'hyperparamètres afin de trouver la meilleure configuration parmi celles proposées. Cependant, cette exploration nécessite un temps de calcul plus important.

En revanche, RandomizedSearchCV réduit considérablement le nombre d'entraînements en sélectionnant aléatoirement un nombre limité de combinaisons. Il peut ainsi obtenir un score très proche de celui de GridSearchCV avec un temps d'exécution inférieur.

Ainsi, GridSearchCV est intéressant lorsque l'on recherche une optimisation exhaustive sur un espace de recherche raisonnable, tandis que RandomizedSearchCV est préférable lorsque l'espace de recherche est vaste et que les ressources de calcul sont limitées.